In [ ]:
!wget https://data.ncl.ac.uk/ndownloader/articles/24574753/versions/1 -O ../data/Gutenberg.zip
!mkdir ../data/Gutenberg
!unzip ../data/Gutenberg.zip -d ../data/Gutenberg/
!rm ../data/Gutenberg.zip

mkdir: ../data/Gutenberg: File exists
unzip:  cannot find or open ../data/Gutenberg.zip, ../data/Gutenberg.zip.zip or ../data/Gutenberg.zip.ZIP.
rm: ../data/Gutenberg.zip: No such file or directory


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
    os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
    torch.set_deterministic_debug_mode(True)



# Настройка пути к корню проекта для импортов
# import sys
# project_root = '/Users/user/prog/nas/self-scaled-optimized-neural-network'
# if project_root not in sys.path:
#     sys.path.insert(0, project_root)



################################
#     RESTART     RUNTIME      #
################################
from ssonn.model.utils import *
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [42]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=27 * 18, output_size=6):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc0(x)

In [43]:
hyperparams = {
    "num_epochs": 128,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.05},
    "pruning_thresholds": {"fc0": 0.01},
    "plateau_threshold": 0.1,
    "min_epochs_between_expansions": 32,
    "plateau_window_size": 5,
    "learning_rate": 5e-4,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 1e-3
}


In [44]:
train_x = np.load("../data/Gutenberg/train_x.npy")
train_y = np.load("../data/Gutenberg/train_y.npy")
valid_x = np.load("../data/Gutenberg/valid_x.npy")
valid_y = np.load("../data/Gutenberg/valid_y.npy")
test_x = np.load('../data/Gutenberg/test_x.npy')
test_y = np.load('../data/Gutenberg/test_y.npy')

train_tensor_x = torch.tensor(train_x).float().view(-1, 27 * 18)
train_tensor_y = torch.tensor(train_y).long()

valid_tensor_x = torch.tensor(valid_x).float().view(-1, 27 * 18)
valid_tensor_y = torch.tensor(valid_y).long()

test_tensor_x = torch.tensor(test_x).float().view(-1, 27 * 18)
test_tensor_y = torch.tensor(test_y).long()

train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
valid_dataset = TensorDataset(valid_tensor_x, valid_tensor_y)
test_dataset = TensorDataset(test_tensor_x, test_tensor_y)

train_loader = DataLoader(train_dataset, batch_size=hyperparams['batch_size'], shuffle=True, generator=g)
val_loader = DataLoader(valid_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)
test_loader = DataLoader(test_dataset, batch_size=hyperparams['batch_size'], shuffle=False, generator=g)

In [45]:
model = SimpleFCN()
sparse_model = convert_dense_to_sparse_network(model, layers=[model.fc0], device=device)

In [ ]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
  api_key="",
  project_name="",
  workspace=""
)
experiment.log_parameters(hyperparams)
experiment.set_name("gutenberg_skip")

COMET INFO: An experiment with the same configuration options is already running and will be reused.


In [47]:
import time
start_time = time.time()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sparse_model.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])
train_sparse_recursive(sparse_model, train_loader, train_loader, val_loader, criterion,optimizer, hyperparams, device, experiment)
print("--- %s seconds ---" % (time.time() - start_time))

100%|██████████| 176/176 [00:00<00:00, 656.64it/s]


------------------------------
Epoch 1/128, Train Loss: 1.7800, Val Loss: 1.7659, Val Accuracy: 0.2447


100%|██████████| 176/176 [00:00<00:00, 763.02it/s]


------------------------------
Epoch 2/128, Train Loss: 1.7507, Val Loss: 1.7455, Val Accuracy: 0.2703


100%|██████████| 176/176 [00:01<00:00, 130.67it/s]


------------------------------
Epoch 3/128, Train Loss: 1.7307, Val Loss: 1.7312, Val Accuracy: 0.2825


... (60 outputs left out)

100%|██████████| 176/176 [00:00<00:00, 784.56it/s]


------------------------------
Epoch 34/128, Train Loss: 1.6550, Val Loss: 1.6799, Val Accuracy: 0.3032
Chosen edges: tensor([[  0,   0,   0,  ...,   5,   5,   5],
        [  0,   1,   2,  ..., 483, 484, 485]]) 1963
[0.0005, 0.0005]


 61%|██████▏   | 108/176 [00:11<00:06,  9.80it/s]


KeyboardInterrupt: 

In [ ]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : gutenberg_skip
COMET INFO:     url                   : https://www.comet.com/shi-van/nas-unseen-dropout/118eb25fb2c7435f9a119e150ed17b86
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     acc amount [64]                    : (4.3695560692442054e-07, 0.00010386374028349338)
COMET INFO:     diff_mean_abs_expand_grads [64]    : (-0.0005887007573619485, 0.002534545725211501)
COMET INFO:     diff_mean_signed_expand_grads [64] : (-2.50430503001553e-07, 1.959356495717657e-06)
COMET INFO:     len_choose                         : 1963
COMET INFO:     lr                                 : 0.0005
COMET INFO:     mean_abs_expand_grads [64]         : (0.001

In [ ]:
_, accuracy = eval_one_epoch(sparse_model, criterion, test_loader, hyperparams['task_type'], device)
params = get_params_amount(sparse_model)

------------------------------


In [ ]:
accuracy, params

(0.41133333333333333, 688634)

In [ ]:
torch.save(sparse_model, 'gutenberg.pt')

In [ ]:
import pandas as pd

data = {
    'Model': ['Ours', 'ResNet-18', 'AlexNet', 'VGG16', 'ConvNext', 'MNASNet', 'DenseNet', 'ResNeXt', 'PC-DARTS', 'DrNAS', 'Bonsai-Net', 'DARTS', ' Bonsai', 'Random'],
    'Accuracy (%)': [accuracy * 100, 49.98, 45.53, 44.00, 31.93, 38.00, 43.28, 40.30, 49.12, 46.62, 48.57, 47.72, 29.00, 16.6],
    'Parameters': [params, 11_689_512, 61_100_840, 138_357_544, 88_591_464, 4_383_312, 28_681_000, 25_028_904, None, None, None, None, None, None]
}

table = pd.DataFrame(data)

def format_with_commas(x):
    return "{:,}".format(x)

styled_table = (table.style
               .format({'Accuracy (%)': '{:.2f}',
                       'Parameters': format_with_commas})
               .set_properties(**{'text-align': 'center'})
               .set_table_styles([
                   {'selector': 'th', 'props': [('text-align', 'center')]},
                   {'selector': 'caption', 'props': [('font-size', '1.1em')]}
               ])
               .hide(axis='index'))

styled_table

Model,Accuracy (%),Parameters
Ours,41.13,"688,634.0"
ResNet-18,49.98,"11,689,512.0"
AlexNet,45.53,"61,100,840.0"
VGG16,44.00,"138,357,544.0"
ConvNext,31.93,"88,591,464.0"
MNASNet,38.00,"4,383,312.0"
DenseNet,43.28,"28,681,000.0"
ResNeXt,40.30,"25,028,904.0"
PC-DARTS,49.12,nan
DrNAS,46.62,nan
